In [3]:
!pip install "numpy<2.0"
# Imports e Modulos
import pandas as pd
import numpy as np
from pgmpy.models import DiscreteBayesianNetwork  
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [4]:
# Topologia da Rede
# Genética e Fumar -> Causa Cancer
# Cancer -> Causa Tosse e altera RaioX
model = DiscreteBayesianNetwork([
    ('Genetica', 'Cancer'),
    ('Fumar', 'Cancer'),
    ('Cancer', 'Tosse'),
    ('Cancer', 'RaioX')
])

In [5]:
# Definição das Tabelas de Probabilidade

# CASOS INDEPENDENTES
# Genética: 10% tem predisposição (1), 90% não (0)
cpd_gen = TabularCPD(variable='Genetica', variable_card=2, values=[[0.90], [0.10]])

# Fumar: 25% fuma (1), 75% não (0)
cpd_fum = TabularCPD(variable='Fumar', variable_card=2, values=[[0.75], [0.25]])

# CASOS DEPENDENTES
# Câncer: O nó central. Depende da combinação de Fumar E Genética.
# Temos 4 combinações possíveis (2x2).
# A ordem das colunas no pgmpy segue a iteração do último para o primeiro nas evidências.
# Evidências: Fumar, Genetica (O inverso da ordem na definição do modelo)
# Colunas:
# 0: Fumar=0, Gen=0 (Baixo Risco)
# 1: Fumar=0, Gen=1 (Risco Médio - só genética)
# 2: Fumar=1, Gen=0 (Risco Médio/Alto - só fumo)
# 3: Fumar=1, Gen=1 (Risco Altíssimo - ambos)

cpd_can = TabularCPD(
    variable='Cancer', variable_card=2,
    values=[
        [0.99, 0.90, 0.70, 0.10], # Prob. NÃO ter Câncer
        [0.01, 0.10, 0.30, 0.90]  # Prob. TER Câncer
    ],
    evidence=['Fumar', 'Genetica'],
    evidence_card=[2, 2]
)

# Tosse: Sintoma clínico.
# Se Cancer=0: 20% tosse (por outros motivos)
# Se Cancer=1: 80% tosse
cpd_tos = TabularCPD(variable='Tosse', variable_card=2, 
                     values=[[0.80, 0.20],  # Não Tosse
                             [0.20, 0.80]], # Tosse
                     evidence=['Cancer'], evidence_card=[2])

# RaioX: Exame de imagem.
# Se Cancer=0: 5% de falso positivo
# Se Cancer=1: 90% de verdadeiro positivo
cpd_rai = TabularCPD(variable='RaioX', variable_card=2, 
                     values=[[0.95, 0.10], # Negativo
                             [0.05, 0.90]], # Positivo
                     evidence=['Cancer'], evidence_card=[2])

In [6]:
# Adicionar e Validar
model.add_cpds(cpd_gen, cpd_fum, cpd_can, cpd_tos, cpd_rai)
assert model.check_model()

In [7]:
# Inferência
infer = VariableElimination(model)

print("--- Cenário 1: Diagnóstico Combinado ---")
# Paciente tem Raio-X positivo E Tosse. Qual a chance de Câncer?
q1 = infer.query(variables=['Cancer'], evidence={'RaioX': 1, 'Tosse': 1}, show_progress=False)
print(f"Chance de Câncer (RaioX+ e Tosse+): {q1.values[1]:.4f}")

print("\n--- Cenário 2: Explaining Away (O Fenômeno do 'Descarte') ---")
# Sabemos que o paciente tem Câncer (hipoteticamente).
# Se descobrirmos que ele tem a Genética ruim, a probabilidade de ele ser Fumante muda?
# Isso é sutil: Se a genética já explica o câncer, a "necessidade" dele ser fumante diminui matematicamente.

print("Chance de Fumar dado apenas que tem Câncer:")
q2a = infer.query(variables=['Fumar'], evidence={'Cancer': 1}, show_progress=False)
print(f"{q2a.values[1]:.4f}")

print("Chance de Fumar dado Câncer E Genética (A causa concorrente):")
q2b = infer.query(variables=['Fumar'], evidence={'Cancer': 1, 'Genetica': 1}, show_progress=False)
print(f"{q2b.values[1]:.4f}")
print("Nota: A probabilidade de fumar caiu pois a Genética já 'explicou' o câncer.")

--- Cenário 1: Diagnóstico Combinado ---
Chance de Câncer (RaioX+ e Tosse+): 0.8934

--- Cenário 2: Explaining Away (O Fenômeno do 'Descarte') ---
Chance de Fumar dado apenas que tem Câncer:
0.8633
Chance de Fumar dado Câncer E Genética (A causa concorrente):
0.7500
Nota: A probabilidade de fumar caiu pois a Genética já 'explicou' o câncer.
